In [40]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from commons import (
    DATASET_CLEAN_LOCATION,
    DATASET_CLEAN_UNDERSAMPLING_LOCATION,
    MODEL_FOLDER,
    VECTORIZERS_FOLDER,
    Datasets,
    vectorize_and_split_dataset,
)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB

# Train Model

Now, we load the datasets that were exported during the data cleaning phase.


In [9]:
df = pd.read_csv(DATASET_CLEAN_LOCATION)
df_undersampling = pd.read_csv(DATASET_CLEAN_UNDERSAMPLING_LOCATION)

I use a utility function from the `commons.py` file, created to avoid duplicating code across notebooks. This function vectorizes the input text using the provided vectorizer and then splits the dataset into training and test sets.

Specifically:

- The function takes a DataFrame containing text and language labels, and a vectorizer (e.g., `CountVectorizer`).
- It transforms the text data into feature vectors with the vectorizer.
- The labels are encoded numerically.
- The dataset is split into training and test subsets with stratification to maintain class distribution.
- The function returns a `Datasets` object that holds the training and test feature matrices, labels, and the original text samples for both sets.

The `Datasets` class is a simple container to keep all these components organized and accessible.



In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from collections import defaultdict
import math
import re
from typing import Iterable
from abc import ABC, abstractmethod

class VectorizerCustom(ABC):
    
    def __init__(self):
        self.vocab = {}
        self.vocab_size = 0
    
    @abstractmethod
    def fit_transform(self, corpus: Iterable[str]) -> csr_matrix:
        pass
    
    @abstractmethod
    def transform(self, corpus: Iterable[str]) -> csr_matrix:
        pass
    
    def get_feature_names_out(self)-> list[str]:
        sorted_vocab = sorted(self.vocab.items(), key=lambda x: x[1])
        return [word for word, idx in sorted_vocab]
    
    def _tokenize(self, text: str)-> list[str]:
        return re.findall(r'\b\w+\b', text)
        # return re.findall(r"(?u)\b\w\w+\b", text)
    

In [67]:
class CountVectorizerCustom(VectorizerCustom):
    def __init__(self):
        super().__init__()
    
    def fit_transform(self, corpus: Iterable[str]) -> csr_matrix:
        rows, cols, data = [], [], []
        vocab = defaultdict(lambda: len(vocab))
        for i, doc in enumerate(corpus):
            word_counts = defaultdict(int)
            for token in self._tokenize(doc):
                word_id = vocab[token]
                word_counts[word_id] += 1
            for word_id, count in word_counts.items():
                data.append(count)
                rows.append(i)
                cols.append(word_id)
        self.vocab = dict(vocab)
        self.vocab_size = len(self.vocab)
        return csr_matrix((data, (rows, cols)), shape=(len(corpus), self.vocab_size))

    def transform(self, corpus: Iterable[str]) -> csr_matrix:
        rows, cols, data = [], [], []

        for i, doc in enumerate(corpus):
            word_counts = defaultdict(int)
            for token in self._tokenize(doc):
                if token in self.vocab:
                    word_id = self.vocab[token]
                    word_counts[word_id] += 1
            for word_id, count in word_counts.items():
                data.append(count)
                rows.append(i)
                cols.append(word_id)

        return csr_matrix((data, (rows, cols)), shape=(len(corpus), self.vocab_size))
    


In [ ]:
class TfidfVectorizerCustom(VectorizerCustom):
    def __init__(self):
        super().__init__()
        self.idf = []


    def fit_transform(self, corpus: Iterable[str]) -> csr_matrix:
        n_docs = len(corpus)
        doc_freq = defaultdict(int)
        vocab = defaultdict(lambda: len(vocab))

        # Prima passata: costruzione vocabolario e idf
        tokenized_docs = []
        for doc in corpus:
            tokens = self._tokenize(doc)
            token_ids = []
            seen = set()
            for token in tokens:
                idx = vocab[token]
                token_ids.append(idx)
                if idx not in seen:
                    doc_freq[idx] += 1
                    seen.add(idx)
            tokenized_docs.append(token_ids)

        self.vocab = dict(vocab)
        self.vocab_size = len(self.vocab)
        self.idf = [
            math.log((1 + n_docs) / (1 + doc_freq[i])) + 1
            for i in range(self.vocab_size)
        ]

        # Seconda passata: calcolo TF-IDF
        rows, cols, data = [], [], []
        for i, token_ids in enumerate(tokenized_docs):
            tf = defaultdict(int)
            for idx in token_ids:
                tf[idx] += 1
            max_tf = max(tf.values())
            for idx, freq in tf.items():
                data.append((freq / max_tf) * self.idf[idx])
                rows.append(i)
                cols.append(idx)

        return csr_matrix((data, (rows, cols)), shape=(n_docs, self.vocab_size))

    def transform(self, corpus: Iterable[str]) -> csr_matrix:
        n_docs = len(corpus)
        rows, cols, data = [], [], []

        for i, doc in enumerate(corpus):
            tf = defaultdict(int)
            tokens = self._tokenize(doc)
            for token in tokens:
                if token in self.vocab:
                    idx = self.vocab[token]
                    tf[idx] += 1
            if not tf:
                continue
            max_tf = max(tf.values())
            for idx, freq in tf.items():
                data.append((freq / max_tf) * self.idf[idx])
                rows.append(i)
                cols.append(idx)

        return csr_matrix((data, (rows, cols)), shape=(n_docs, self.vocab_size))


In [69]:
vectorizer_bow = CountVectorizerCustom()
vectorizer_bow_und = CountVectorizerCustom()
vectorizer_tfidf = TfidfVectorizerCustom()
vectorizer_tfidf_und = TfidfVectorizerCustom()

df_bow = vectorize_and_split_dataset(df, vectorizer_bow)
df_bow_undersampling = vectorize_and_split_dataset(df_undersampling, vectorizer_bow_und)
df_tfidf = vectorize_and_split_dataset(df, vectorizer_tfidf)
df_tfidf_undersampling = vectorize_and_split_dataset(df_undersampling, vectorizer_tfidf_und)

I have chosen **Multinomial Naive Bayes (MNB)** and **Logistic Regression (LR)** as our classification algorithms based on insights from the following research articles:

- [Language Identification Using Multinomial Naive Bayes Technique](https://www.researchgate.net/publication/377067809_Language_Identification_Using_Multinomial_Naive_Bayes_Technique)
- [Language Identification Using Combination of Machine Learning Algorithms and Vectorization Techniques](https://www.researchgate.net/publication/362096783_Language_Identification_Using_Combination_of_Machine_Learning_Algorithms_and_Vectorization_Techniques)

**Reason for choosing these models:**

- **Multinomial Naive Bayes:**  
  This model is widely used in text classification tasks due to its simplicity, efficiency, and strong performance, especially when features represent term frequencies. The first article highlights how MNB effectively captures the distribution of words in different languages, making it a natural fit for language identification.

- **Logistic Regression:**  
  Logistic Regression is a robust, interpretable linear model that often performs well on binary and multiclass classification problems. According to the second article, combining logistic regression with appropriate vectorization techniques can improve classification accuracy.


In [70]:
def naive_bayes(datasets: Datasets) -> MultinomialNB:
    """
    Trains and evaluates a Naive Bayes classifier on the provided dataset.

    This function fits a Multinomial Naive Bayes model using the training data,
    evaluates its accuracy on the test set, and prints the accuracy score as well
    as a summary of misclassified examples including the original text, true label,
    and predicted label.

    Args:
        datasets (Datasets): A Datasets object containing:
            - X (features for training)
            - y (labels for training)
            - X_t (features for testing)
            - y_t (labels for testing)
            - text_t (original text corresponding to test samples)

    Returns:
        MultinomialNB: The trained Naive Bayes classifier.

    """
    nb = MultinomialNB()
    nb.fit(datasets.X, datasets.y)
    y_pred = nb.predict(datasets.X_t)
    print("Naive Bayes:", accuracy_score(datasets.y_t, y_pred))

    wrong_idx = datasets.y_t != y_pred
    errors_df = pd.DataFrame({
        "Text": datasets.text_t[wrong_idx],
        "True Label": datasets.y_t[wrong_idx],
        "Predicted Label": y_pred[wrong_idx],
    })
    for _, row in errors_df.iterrows():
        print(f"📝 Text: {row['Text']}")
        print(f"✅ True Label: {row['True Label']}")
        print(f"❌ Predicted: {row['Predicted Label']}")
        print("-" * 50)
    return nb


In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

class LogisticRegressionSparse:
    def __init__(self, learning_rate=0.1, epochs=1000):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.w = None
        self.b = 0

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X: csr_matrix, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.epochs):
            z = X.dot(self.w) + self.b
            y_pred = self.sigmoid(z)
            error = y_pred - y
            dw = X.T.dot(error) / n_samples 
            db = np.sum(error) / n_samples
            self.w -= self.learning_rate * dw
            self.b -= self.learning_rate * db

    def predict_proba(self, X: csr_matrix):
        z = X.dot(self.w) + self.b
        return self.sigmoid(z)

    def predict(self, X: csr_matrix):
        return (self.predict_proba(X) >= 0.5).astype(int)


In [72]:
lr = LogisticRegressionSparse(epochs=1000)
lr.fit(df_tfidf.X, df_tfidf.y)

In [73]:
lr

In [74]:
def logistic_regression(datasets: Datasets) -> LogisticRegressionSparse:
    """
    Trains and evaluates a logistic regression classifier using grid search with cross-validation.

    This function performs hyperparameter tuning on a logistic regression model using a predefined
    parameter grid and 5-fold cross-validation. It trains the model on the provided training data,
    evaluates accuracy on the test set, and prints the best parameters, cross-validation accuracy,
    and misclassified examples.

    Args:
        datasets (Datasets): A Datasets object containing:
            - X (features for training)
            - y (labels for training)
            - X_t (features for testing)
            - y_t (labels for testing)
            - text_t (original text corresponding to test samples)

    Returns:
        GridSearchCV: The fitted GridSearchCV object containing the best estimator.

    """
    grid = LogisticRegressionSparse(epochs=1000, learning_rate=1000)
    grid.fit(datasets.X, datasets.y)


    y_pred = grid.predict(datasets.X_t)
    print("Logistic Regression:", accuracy_score(datasets.y_t, y_pred))

    wrong_idx = datasets.y_t != y_pred
    errors_df = pd.DataFrame({
        "Text": datasets.text_t[wrong_idx],
        "True Label": datasets.y_t[wrong_idx],
        "Predicted Label": y_pred[wrong_idx],
    })
    for _, row in errors_df.iterrows():
        print(f"📝 Text: {row['Text']}")
        print(f"✅ True Label: {row['True Label']}")
        print(f"❌ Predicted: {row['Predicted Label']}")
        print("-" * 50)

    return grid

We try the Multinomial Naive Bayes and observe that the inputs with undersampling perform better for Bag of Words vectorizations.


I display the predicted probabilities from the Naive Bayes model to highlight an important issue: some words (such as "in") appear in multiple languages, which can confuse the classifier and affect its accuracy.

We also try Logistic Regression and observe that the models trained without undersampling perform better with TF-IDF vectorizations in terms of accuracy. However, this improvement is based solely on accuracy, so other metrics should be carefully analyzed to get a more complete understanding of model performance.

In [75]:
nb_bow = naive_bayes(df_bow)

Naive Bayes: 0.9709724238026125
📝 Text: ഇഗലഷ വകകപഡയയൽ പലപപഴ ഭരപകഷ ആളകളട അഭപരയ സതയ എനന രപതതൽ അടചചൽപപകകൻ സദധയതയണടവറണട ഉദഹരണതതന കശമർ പരശന ഇതൽ പകസതൻ വശജരകകള കടതൽ വകകപഡയ ഉപയകതകകൾ ഇനതയയൽ നനന ഉളളവരയതനൽ ലഖനതതന ഇനതയ അനകല ചയവ വരൻ സദധയതയണട
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: തങങളട ഇടയൽ നഴഞഞകയറയടടണടയകകവനന വകകവരദധര ഭയനന സമഹതതന നരടട തരതതൻ കഴയതത രതയലകക നർദദശചചടടളള തളകളൽ മററ വരതതവൻ വകകപഡയർ കരയനർവവഹകര നയഗചചരകകനന
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: ജനകയ പങകളതതതതലട മതവബസററനകകൾ പരശസത കവരകകൻ വകകപഡയയകക സധചച
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: владелец сайта американская некоммерческая организация фонд викимедиа имеющая региональных представительств
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: எனனம இதன மறததரதத நசசர தனத தரபபல இதறகன மறயன அறககயயம பரடடனககவன தலமயன மறபபககளககன எதர வதஙகளயம வளயடடத
✅ True Label: 0


In [81]:
nb_bow_und = naive_bayes(df_bow_undersampling)

Naive Bayes: 0.9927007299270073
📝 Text: progettando
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: con reparar algo
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------


In [77]:
feature_names = vectorizer_bow.get_feature_names_out()
log_prob = nb_bow.feature_log_prob_
prob_not_it = np.exp(log_prob[0])
prob_it = np.exp(log_prob[1])
df_prob_nb = pd.DataFrame({
    "word": feature_names,
    "P(word|not it)": prob_not_it,
    "P(word|it)": prob_it,
})

print("Most Important words for not italian class:")
print(df_prob_nb.sort_values("P(word|not it)", ascending=False).head(10))
print("Most Important words for  italian class:")
print(df_prob_nb.sort_values("P(word|it)", ascending=False).head(10))

Most Important words for not italian class:
      word  P(word|not it)  P(word|it)
12396   de        0.014374    0.000017
22       a        0.006816    0.002914
2      the        0.006681    0.000086
10638   en        0.006645    0.000017
8136     क        0.006080    0.000017
12413  que        0.005329    0.000017
3761    la        0.005183    0.003377
8158     ह        0.004644    0.000017
16      of        0.004572    0.000069
1       in        0.003841    0.003909
Most Important words for  italian class:
      word  P(word|not it)  P(word|it)
31740   di        0.000005    0.009240
562      e        0.001721    0.005040
31764  che        0.000005    0.004543
1       in        0.003841    0.003909
31727    è        0.000005    0.003669
15915   un        0.002457    0.003429
3761    la        0.005183    0.003377
15895   il        0.000472    0.003240
22       a        0.006816    0.002914
15906  non        0.000104    0.002606


In [82]:
lr_tfidf = logistic_regression(df_tfidf)

Logistic Regression: 0.9956458635703919
📝 Text: hai assolutamente ragione
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: a dopo
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: posso offrirti un bicchiere d acqua
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: tu sei forte
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: dita incrociate
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: dai
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: sei libero sabato prossimo
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: non ne vale la pena
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: lo sai
✅ True Label: 1
❌ Predicted: 0
---------------------------------------------

In [83]:
lr_tfidf_und = logistic_regression(df_tfidf_undersampling)

Logistic Regression: 0.9744525547445255
📝 Text: hey guys welkom op mijn kanaal in deze video
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: y escuche la pronunciación una o dos veces
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: progettando
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: embora em geral elogiando o artigo considerou sua conclusão indecisa alguns historiadores lembramse dele como um foradalei oportunista e sedento de sangue enquanto outros continuam a vêlo como um soldado audaz e um herói popular
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: à terme il parvint à battre le e meilleur joueur des étatsunis
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: penny arcade in bir karikatüründe iskeletor heman maddesini değiştirmektedir
✅ True Label: 0
❌ Predicted: 1
--

After evaluating the performance of both Multinomial Naive Bayes (MNB) and Logistic Regression (LR) classifiers with various vectorization methods and sampling strategies, I observed mixed results. While Logistic Regression trained on the original, imbalanced dataset performed better in accuracy, the undersampled version with MNB and Bag-of-Words (BoW) showed promising results. Therefore, for the next phase of the analysis, I decided to continue analyzing all the 4 combinations to evaluate other metrics



In [84]:
PATH_MODEL_FOLDER = Path(MODEL_FOLDER)
with (PATH_MODEL_FOLDER / "nb_bow.pkl").open("wb") as f:
    pickle.dump(nb_bow, f)

with (PATH_MODEL_FOLDER / "nb_bow_und.pkl").open("wb") as f:
    pickle.dump(nb_bow_und, f)

with (PATH_MODEL_FOLDER / "lr_tfidf.pkl").open("wb") as f:
    pickle.dump(lr_tfidf, f)

with (PATH_MODEL_FOLDER / "lr_tfidf_und.pkl").open("wb") as f:
    pickle.dump(lr_tfidf_und, f)

I save the vectorizers so that I can use them later during the inference phase.


In [85]:
PATH_VECTORIZERS_FOLDER = Path(VECTORIZERS_FOLDER)

with (PATH_VECTORIZERS_FOLDER / "vectorizer_bow.pkl").open("wb") as f:
    pickle.dump(vectorizer_bow , f)

with (PATH_VECTORIZERS_FOLDER / "vectorizer_bow_und.pkl").open("wb") as f:
    pickle.dump(vectorizer_bow_und , f)

with (PATH_VECTORIZERS_FOLDER / "vectorizer_tfidf.pkl").open("wb") as f:
    pickle.dump(vectorizer_tfidf , f)

with (PATH_VECTORIZERS_FOLDER / "vectorizer_tfidf_und.pkl").open("wb") as f:
    pickle.dump(vectorizer_tfidf_und , f)